## INSTALLING DEPENDENICES
RAG pipeline:
document loading
embedding models
vector database
llm model


In [5]:
print("Installing Dependencies....")

%pip install --user langchain \
   langchain-community \
   langchain-text-splitters \
   langchain-huggingface \
   langchain-groq \
   langchain-google-genai \
   langchain-openai \
   langchain-core \
   faiss-cpu \
   pypdf \
   sentence-transformers \
   transformers \
   torch \
   huggingface-hub \
   qdrant-client \
   langsmith \
   python-dotenv \
   tiktoken

print('\Installation Complete!')

Installing Dependencies....


<>:5: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
<>:5: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
C:\Users\lenovo\AppData\Local\Temp\ipykernel_2148\1717814941.py:5: SyntaxWarning: "\I" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\I"? A raw string is also an option.
  print('\Installation Complete!')


PermissionError: [WinError 5] Access is denied

In [95]:
LLM_Provider = 'groq'
LLM_Model = 'openai/gpt-oss-20b'


CORPOUS_PATH = "./zyro-dynamic-hr-corpous/" #<-- Knowledge Base 



In [96]:
import os
from dotenv import load_dotenv
load_dotenv()
if os.getenv("GROQ_API_KEY"):
    print("API KEY IS IMPORTED")
else:
    print("API KEY NOT FOUND")
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 


API KEY IS IMPORTED


In [97]:
import glob
from langchain_community.document_loaders import PyPDFLoader

# Note the 's' in zyro-dynamics
CORPUS_PATH = "./zyro-dynamics-hr-corpus"

# Find all PDF files inside the main folder and all subfolders
pdf_files = glob.glob(f"{CORPUS_PATH}/**/*.pdf", recursive=True)

documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(pdf_path)
    documents.extend(loader.load())

print(f"Loaded {len(documents)} documents from {len(pdf_files)} PDF files.")


Loaded 39 documents from 11 PDF files.


In [98]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 100,
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")


Created 107 chunks


## Embeddings 
 hugging face embeddings
 

In [99]:
from langchain_huggingface import HuggingFaceEmbeddings
model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)
print("Embeddings done!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2486.22it/s]


Embeddings done!


## vectordb+Retrival
Faiss vector DB


In [100]:
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks , model) # CHUNKS of data and embedding model

#retrival 
retriever = vectorstore.as_retriever(search_kwargs = {"k" :3})
print("Done")

Done


## LLM Initialization
LLM model - groq


In [101]:
from langchain_groq import ChatGroq
llm_model = ChatGroq(
model = LLM_Model,
temperature =0.7,
max_tokens = 500,
groq_api_key = LLM_API_KEY
)
print(f"LLM Model groq initialized")



LLM Model groq initialized


## RAG CHAIN
initialization for entire Rag Chain


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

RAG_PROMPT = ChatPromptTemplate.from_template(
    ## You define your prompt
    """ 
    You are an HR assistance. Answers the question using ONLY 
    the context below. If the answers isn't in the context,
      say you don't have that information.

    context : {context}, 
    Question : {question}"""
)
def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)



@traceable(name="rag_chain")
def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs) # all the info req for llm to get output
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer = chain.invoke({"context" : context , "Question" : question})
    return {"answer" : answer }

    


## Guardials 
(to ensure no context is used beyond the corpus)

In [105]:

GUARDIAL_PROMPT = ChatPromptTemplate.from_template("""
You are a scope classifier for an HR assistant. Decide whether the question below is something an HR assistant should answer (company leave policy,
reimbursement, code of conduct , or other internal HR topics) or something out of scope (general knowledge , coding help , unrelated small task, etc).
Response with exactly one word: IN_SCOPE or OUT_OF_SCOPE.
Question: {question}
""")
REFUSAL_MESSAGE = (
    "I'm an HR assistance and can only help with questions about company HR "
    "policies (leave , reimbursement , code of conduct, etc.). I don't have"
    "information to answer that question."
)

def ask_bot(question: str):
    guardial_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict = guardial_chain.invoke("question : {question}").strip().upper()
    if 'OUT_OF_SCOPE' in verdict:
        return {'answer' : REFUSAL_MESSAGE , "sources" : []}
    return rag_chain(question)
print("Guardial Initialized")


    



Guardial Initialized


In [106]:
test_question =[
    "How many casual leaves do I get per year?" , 
    "What is the internet reimbursement limit?" , 
    "What's the capital of France?",
]
for i,q in enumerate(test_question , 1):
    result = ask_bot(q)
    print(f"Q{i} : {q}")
    print(f"A{i} : {result['answer']}")

Q1 : How many casual leaves do I get per year?
A1 : I'm an HR assistance and can only help with questions about company HR policies (leave , reimbursement , code of conduct, etc.). I don't haveinformation to answer that question.
Q2 : What is the internet reimbursement limit?
A2 : I'm an HR assistance and can only help with questions about company HR policies (leave , reimbursement , code of conduct, etc.). I don't haveinformation to answer that question.
Q3 : What's the capital of France?
A3 : I'm an HR assistance and can only help with questions about company HR policies (leave , reimbursement , code of conduct, etc.). I don't haveinformation to answer that question.
